In [1]:
import numpy as np
from scipy.special import jv, jn_zeros
from scipy.sparse.linalg import eigsh, ArpackNoConvergence
from scipy.sparse import diags
from scipy.interpolate import UnivariateSpline
from scipy.integrate import quad
from joblib import Parallel, delayed
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# =============================================
# 1. Parámetros Físicos Optimizados
# =============================================
D0 = 2e-9           # Coeficiente de difusión [m²/s]
R = 1e-6            # Radio del cilindro [m]
z_c = 10e-9         # Altura crítica [m]
lambda_param = 0.2  # Parámetro de decaimiento
delta = 1e-9        # Ancho de transición
n_modes_radial = 8  # Modos radiales
m_max = 10          # Modos axiales
N_z = 600           # Nodos en z
alpha_grid = 8.0    # Parámetro de malla
z0 = 15e-9          # Posición axial inicial [m]
rho0 = 0.75e-6      # Posición radial inicial [m]

# =============================================
# 2. Función de Difusión (Corregida)
# =============================================
def D(z_val):
    """Perfil de difusión corregido"""
    S = 0.5*(1 + np.tanh((z_val - z_c)/delta))
    return D0*(np.exp(-lambda_param*z_val)*(1 - S) + S)

# =============================================
# 3. Generación de Malla Adaptativa
# =============================================
j = np.linspace(0, 1, N_z)
z = z_c * (1 + np.sinh(alpha_grid*(j - 0.5)) / np.sinh(0.5*alpha_grid))
D_values = D(z)

# =============================================
# 4. Solución Radial (Funciones de Bessel)
# =============================================
alpha_zeros = jn_zeros(0, n_modes_radial)
alpha_n = alpha_zeros / R

# =============================================
# 5. Solución Axial Paralelizada (Corregida)
# =============================================
def build_system(n):
    try:
        alpha = alpha_n[n]
        h = np.diff(z)
        N = len(z)
        
        main_diag = np.zeros(N)
        upper_diag = np.zeros(N-1)
        lower_diag = np.zeros(N-1)
        
        for j in range(1, N-1):
            h_prev = h[j-1]
            h_next = h[j]
            
            D_prev = 0.5*(D_values[j] + D_values[j-1])
            D_next = 0.5*(D_values[j] + D_values[j+1])
            
            main_diag[j] = D_prev/h_prev + D_next/h_next + alpha**2*D_values[j]
            upper_diag[j] = -D_next/h_next
            lower_diag[j-1] = -D_prev/h_prev
        
        # Condiciones de frontera
        main_diag[0] = D_values[0]/h[0] + 1e-10*D0
        upper_diag[0] = -D_values[0]/h[0]
        main_diag[-1] = 1.0
        lower_diag[-1] = -1.0
        
        A = diags([main_diag, upper_diag, lower_diag], [0, 1, -1], format='csr')
        eigenvalues, eigenvectors = eigsh(A, k=m_max, which='SA', tol=1e-6)
        return (n, (eigenvalues, eigenvectors))
    
    except ArpackNoConvergence:
        print(f"Modo radial {n} no convergió. Usando ceros.")
        return (n, (np.zeros(m_max), np.zeros((N_z, m_max))))

# Ejecución paralela
results = Parallel(n_jobs=4)([delayed(build_system)(n) for n in range(n_modes_radial)])

# Procesamiento robusto de resultados
eigenvalues_dict = {}
eigenvectors_dict = {}
for result in results:
    n = result[0]
    evals, evecs = result[1]
    
    # Asegurar que evecs sea 2D
    if evecs.ndim == 1:
        evecs = evecs.reshape(-1, 1)  # Convertir a 2D
        
    eigenvalues_dict[n] = evals
    eigenvectors_dict[n] = evecs

# =============================================
# 6. Interpolación (Corregida)
# =============================================
Z_interps = {}
for n in range(n_modes_radial):
    m_max_real = eigenvectors_dict[n].shape[1]  # Número real de modos
    Z_interps[n] = [
        UnivariateSpline(z, eigenvectors_dict[n][:, m], s=0, ext='zeros')
        for m in range(m_max_real)
    ]

# =============================================
# 7. Cálculo de Coeficientes A_nm (Corregido)
# =============================================
def compute_A_nm(n):
    Z_norms = np.zeros(m_max)
    for m in range(m_max):
        integrand = lambda z_val: (Z_interps[n][m](z_val)**2) / D(z_val)
        Z_norms[m] = quad(integrand, z[0], z[-1], limit=500, epsabs=1e-20)[0]
    
    Z_m_z0 = np.array([Z_interps[n][m](z0) for m in range(m_max)])
    J0_rho0 = jv(0, alpha_n[n]*rho0)
    
    valid = (Z_norms > 1e-25) & (np.abs(Z_m_z0) > 1e-12)
    return (J0_rho0 * Z_m_z0 * valid) / (2*np.pi*rho0 * (Z_norms + 1e-30))

A_nm = Parallel(n_jobs=4)([delayed(compute_A_nm)(n) for n in range(n_modes_radial)])

# =============================================
# 8. Densidad 3D Corregida (Vectorizada)
# =============================================
def plot_density_3d(t, n_points=40):
    rho_vals = np.linspace(0.1*R, 0.9*R, n_points)
    z_vals = np.linspace(5e-9, 25e-9, n_points)  # Rango extendido
    Rg, Zg = np.meshgrid(rho_vals, z_vals)
    
    # Precalcular términos
    decay = np.array([np.exp(-eigenvalues_dict[n]*t) for n in range(n_modes_radial)])
    A_nm_arr = np.array(A_nm)
    
    # Evaluación vectorizada
    P = np.zeros_like(Rg)
    for n in range(n_modes_radial):
        J0 = jv(0, alpha_n[n]*Rg)
        Z_m = np.array([Z_interps[n][m](Zg) for m in range(m_max)])
        contrib = np.sum(A_nm_arr[n][:, None, None] * J0 * Z_m * decay[n][:, None, None], axis=0)
        P += np.abs(contrib)
    
    # Normalización y estabilidad
    P = np.nan_to_num(P, posinf=0.0, neginf=0.0)
    P = np.clip(P, 0, 1e8)
    
    # Gráfico
    fig = plt.figure(figsize=(12, 8))
    ax = fig.add_subplot(111, projection='3d')
    surf = ax.plot_surface(Rg*1e6, Zg*1e9, P, cmap='viridis', 
                          rstride=1, cstride=1, alpha=0.8)
    ax.set_xlabel(r'$\rho$ (μm)')
    ax.set_ylabel('z (nm)')
    ax.set_zlabel('P(ρ,z,t)')
    plt.title(f'Densidad en t = {t:.1e} s')
    plt.savefig(f'densidad_3d_t_{t:.0e}.pdf')
    plt.close()

# Ejemplos de uso
plot_density_3d(1e-9)
plot_density_3d(1e-6)
plot_density_3d(1e-3)

# =============================================
# 9. Cálculo de G(t) Vectorizado (Corregido)
# =============================================
def compute_G(t, N_quad=30):
    # Configuración de nodos extendidos
    rho_nodes = np.linspace(0.2*R, 0.8*R, N_quad)
    z_nodes = np.linspace(5e-9, 25e-9, N_quad)
    
    # Malla vectorizada
    Rg, Zg = np.meshgrid(rho_nodes, z_nodes)
    R_flat = Rg.flatten()
    Z_flat = Zg.flatten()
    weights = np.outer(np.ones_like(rho_nodes), np.ones_like(z_nodes)).flatten()
    
    # Términos precalculados
    r_mag = np.sqrt(R_flat**2 + Z_flat**2 + 1e-30)
    f_values = (3*(Z_flat**2) - R_flat**2) / (r_mag**5)
    
    total = 0.0
    for n in range(n_modes_radial):
        # Evaluación vectorizada
        J0 = jv(0, alpha_n[n]*R_flat)
        Z_m = np.array([Z_interps[n][m](Z_flat) for m in range(m_max)])
        decay = np.exp(-eigenvalues_dict[n][None, :] * t)
        
        contrib = np.sum(A_nm[n][:, None] * J0 * Z_m * decay, axis=1)
        total += np.sum(weights * f_values * contrib)
    
    return (5/(16*np.pi)) * np.clip(total, 1e-40, 1e10)

# Cálculo de G(t)
t_values = np.logspace(-9, -2, 25)
G_values = Parallel(n_jobs=4)([delayed(compute_G)(t) for t in t_values])

# Guardar resultados
np.savetxt('Gt_corregido.dat', np.column_stack((t_values, G_values)), 
           header='Tiempo(s) G(t)', fmt='%.6e')

# Gráfico final
plt.figure(figsize=(10, 6))
plt.loglog(t_values, G_values, 'o-', markersize=6, linewidth=1.5)
plt.xlabel('Tiempo (s)')
plt.ylabel('G(t)')
plt.title('Función de Correlación Nano-NMR Corregida')
plt.grid(which='both', alpha=0.4)
plt.savefig('Gt_corregido.pdf')
plt.show()

ValueError: operands could not be broadcast together with shapes (10,900) (1,10) 